In [43]:
import json
import pandas as pd
from google import genai
from google.genai import types

In [44]:
MODEL_NAME = "gemini-3.5-flash-lite"

SCHEMA = {
    "type": "object",
    "properties": {
        "root_cause": {
            "type": "string"
        },
        "confidence": {
            "type": "number"
        },
        "summary": {
            "type": "string"
        },
        "recommended_action": {
            "type": "string"
        }
    },
    "required": [
        "root_cause",
        "confidence",
        "summary",
        "recommended_action"
    ]
}

SYSTEM_TEXT = """
You are a clinical note analysis assistant.
Analyse the provided clinical note carefully.
Do not invent information that is not present in the note.
Return only valid JSON matching the provided schema.
"""

In [45]:
def build_prompt(note_text):

    prompt = f"""
    Context:
    You are analysing clinical notes to identify the root cause
    of a patient's issue.

    Role:
    Act as a clinical data analyst.

    Action:
    Analyse the clinical note and identify its root cause.

    Allowed root_cause values:
    - scheduling_error
    - patient_cancellation
    - transport_issue
    - provider_unavailable
    - other

    Determine a confidence score between 0 and 1.
    Write a summary containing at most 25 words.
    Recommend an appropriate action based on the note.

    Tone:
    Professional, objective and concise.

    Format:
    Return JSON with exactly these fields:
    root_cause, confidence, summary, recommended_action.

    Clinical note:
    {note_text}
    """

    return prompt

In [ ]:
API_KEY = input("Paste your Google AI Studio API key: ")
client = genai.Client(api_key=API_KEY)

In [59]:
def analyse_note(note_text):

    prompt = build_prompt(note_text)

    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_TEXT,
            max_output_tokens=1500,
            response_mime_type="application/json",
            response_schema=SCHEMA,
            thinking_config=types.ThinkingConfig(
                thinking_level="minimal"
            )
        )
    )

    return response.text

In [51]:
file_name = '/content/ecWvPsdV2FNmRZAa.csv'

notes_df = pd.read_csv(file_name)

print(notes_df.head())

  note_id appointment_id                                          note_text
0    N001          A0872  Patient said she never got any reminder messag...
1    N002          A0897  Lives far from the clinic and could not find a...
2    N003          A0526  Patient felt the consultation fee was too high...
3    N004          A0926                                                NaN
4    N005          A0534  Had high fever and body ache on the day, staye...


In [60]:
note_n001 = notes_df.loc[
    notes_df["note_id"] == "N001",
    "note_text"
].iloc[0]

result = analyse_note(note_n001)

print(result)

{
  "root_cause": "scheduling_error",
  "confidence": 0.95,
  "summary": "Patient forgot the appointment because she did not receive a reminder message and requested to rebook for next week.",
  "recommended_action": "Reschedule the appointment for next week and review reminder notification settings."
}


In [49]:
print(MODEL_NAME)

gemini-3.5-flash-lite


PART 4.2

In [61]:
from pydantic import BaseModel, Field

class FollowupInsight(BaseModel):

    root_cause: str = Field(min_length=3)
    confidence: float = Field(ge=0, le=1)
    summary: str = Field(min_length=5)
    recommended_action: str = Field(min_length=5)

In [62]:
ALLOWED_ROOT_CAUSES = [
    "scheduling_error",
    "patient_cancellation",
    "transport_issue",
    "provider_unavailable",
    "other"
]

In [64]:
result = json.loads(analyse_note(note_n001))
print(result)

{'root_cause': 'scheduling_error', 'confidence': 0.95, 'summary': 'Patient forgot the appointment because she did not receive a reminder message and requested to rebook for next week.', 'recommended_action': "Reschedule the patient's visit for next week and verify her contact information for future reminders."}


In [65]:
import time

def analyse_note_with_retry(note_text, max_retries=4):

    for attempt in range(max_retries):
        try:
            return analyse_note(note_text)

        except Exception as e:

            if "503" in str(e) or "UNAVAILABLE" in str(e):

                wait_time = 20 * (attempt + 1)

                print(f"Gemini busy. Waiting {wait_time} seconds...")

                time.sleep(wait_time)

            else:
                raise

    raise Exception("Gemini unavailable after maximum retries")

In [ ]:
results = []

api_calls = 0

for index, row in notes_df.head(10).iterrows():

    note_text = row["note_text"]

    # Default output
    result = {
        "note_id": row["note_id"],
        "appointment_id": row["appointment_id"],
        "root_cause": "",
        "confidence": "",
        "summary": "",
        "recommended_action": "",
        "validation_status": "",
        "raw_response": ""
    }

    # 1. Check blank or missing notes FIRST
    if pd.isna(note_text) or str(note_text).strip() == "":

        result["root_cause"] = "other"
        result["confidence"] = 0.0
        result["summary"] = "No note recorded"
        result["recommended_action"] = "Re-call patient"
        result["validation_status"] = "skipped_blank"

    else:

        try:
            # 2. Pause between API calls
            if api_calls > 0:
                time.sleep(13)

            api_calls += 1

            # 3. Call Gemini
            raw_response = analyse_note_with_retry(str(note_text))

            # Preserve exact response
            result["raw_response"] = raw_response

            # 4. Parse JSON
            data = json.loads(raw_response)

            # 5. Copy whichever fields are present
            for field in [
                "root_cause",
                "confidence",
                "summary",
                "recommended_action"
            ]:
                if field in data:
                    result[field] = data[field]

            # 6. Validate using Pydantic
            insight = FollowupInsight(**data)

            # 7. Check allowed root cause
            if insight.root_cause not in ALLOWED_ROOT_CAUSES:
                raise ValueError("Invalid root_cause")

            result["validation_status"] = "valid"

        except Exception as e:

            result["validation_status"] = "invalid"

            print(f"Error processing {row['note_id']}: {e}")

    # 8. Keep every row, valid or invalid
    results.append(result)

Gemini busy. Waiting 20 seconds...
Gemini busy. Waiting 20 seconds...
Gemini busy. Waiting 40 seconds...
Gemini busy. Waiting 60 seconds...


In [57]:
for field in [
    "root_cause",
    "confidence",
    "summary",
    "recommended_action"
]:
    if field in data:
        result[field] = data[field]

In [ ]:
output_df = pd.DataFrame(results)

output_df = output_df[
    [
        "note_id",
        "appointment_id",
        "root_cause",
        "confidence",
        "summary",
        "recommended_action",
        "validation_status",
        "raw_response"
    ]
]

output_df.to_csv(
    "/content/notes_analysed.csv",
    index=False
)

print(output_df.head(10))